# 🧱 Lab: Prompt A/B Evaluation

**Module 9: Is It Any Good? Quality, Latency, Cost** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Name** a handful of production failure modes that evaluation should watch for
2. **Run** prompt V1 vs V2 on the gold set
3. **Write** a one-paragraph ship / don't-ship decision with a self-check rubric
4. **Tag** `v4`

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Prompt versioning | Messages, roles, params are structured data with a version label | mini-typed-prompt-parser |
| Gold set | Small verified labelled sample | mini-score-against-gold-set |
| Precision and recall | Per-class beats headline | mini-score-against-gold-set |
| Baseline comparison | Judge against a simple reference | mini-score-against-gold-set |
| Evidence-based decision | "I would ship V2 because ___, accepting ___" | This lab |
| AI failure modes | Out-of-schema labels, drift, rate limits, silent fallback | This lab |

## Prerequisites

- **mini-score-against-gold-set**: Baseline + gold set scoring
- **mini-latency-and-cost**: p50 / p95 and cost per 1,000


---
## Production failure modes (short lecture)

Watch for: out-of-schema labels, silent fallback, rate limits, input drift, and a gold set that is too small to trust a 2-point accuracy bump. Name them in `EVALUATION.md` when you write the ship sentence.


## The Scenario

You have an LLM-based system that classifies support tickets into three categories: **billing**, **technical**, and **account**. Your team wrote a prompt (V1) that's been running in production. A colleague proposed a rewritten prompt (V2) that they claim is better.

**Your job:** Run both prompts against the same gold set, measure quality + latency + cost, and write a defensible recommendation — ship V2, keep V1, or neither.

This is **prompt versioning** in practice: treating prompt changes like code changes and evaluating them with evidence before deploying.

In [1]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Step 1 — The Gold Set and Two Prompt Versions

We have a 40-row gold set of support tickets with human-verified labels. Both prompt V1 and V2 were run against every ticket. We also logged latency and token usage for each call (as we learned in `mini-latency-and-cost`).

In a real project, you'd run the actual LLM calls. Here we use the recorded results so the lab is reproducible and doesn't require an API key.

In [2]:
# ---------- Gold set: 40 tickets with human-verified labels ----------
gold_labels = [
    # 18 billing
    "billing","billing","billing","billing","billing","billing",
    "billing","billing","billing","billing","billing","billing",
    "billing","billing","billing","billing","billing","billing",
    # 13 technical
    "technical","technical","technical","technical","technical",
    "technical","technical","technical","technical","technical",
    "technical","technical","technical",
    # 9 account
    "account","account","account","account","account",
    "account","account","account","account",
]

# ---------- Prompt V1 predictions ----------
# V1: a short, zero-shot prompt. Good at billing, weaker on technical/account.
v1_predictions = [
    # billing (18): 15 correct, 2→technical, 1→account
    "billing","billing","billing","billing","billing","billing",
    "billing","billing","billing","billing","billing","billing",
    "billing","billing","billing","technical","technical","account",
    # technical (13): 8 correct, 3→billing, 2→account
    "technical","technical","technical","technical","technical",
    "technical","technical","technical","billing","billing",
    "billing","account","account",
    # account (9): 5 correct, 3→billing, 1→technical
    "account","account","account","account","account",
    "billing","billing","billing","technical",
]

# ---------- Prompt V2 predictions ----------
# V2: a longer prompt with examples and definitions. Better on technical/account,
# but slightly worse on billing (the added length occasionally confuses it).
v2_predictions = [
    # billing (18): 14 correct, 2→technical, 2→account
    "billing","billing","billing","billing","billing","billing",
    "billing","billing","billing","billing","billing","billing",
    "billing","billing","technical","technical","account","account",
    # technical (13): 11 correct, 1→billing, 1→account
    "technical","technical","technical","technical","technical",
    "technical","technical","technical","technical","technical",
    "technical","billing","account",
    # account (9): 7 correct, 1→billing, 1→technical
    "account","account","account","account","account",
    "account","account","billing","technical",
]

from collections import Counter
dist = Counter(gold_labels)
print(f"Gold set size: {len(gold_labels)} tickets")
print(f"Label distribution: {dict(dist)}")
print(f"V1 predictions: {len(v1_predictions)}, V2 predictions: {len(v2_predictions)}")

Gold set size: 40 tickets
Label distribution: {'billing': 18, 'technical': 13, 'account': 9}
V1 predictions: 40, V2 predictions: 40


## Step 2 — Majority-Class Baseline

Recall from `mini-run-majority-baseline`: before comparing V1 and V2, we need to know the accuracy of the *dumbest possible system* — one that always predicts the most common label. Both prompts must beat this to justify their existence.

In [3]:
def accuracy(true, predicted):
    """Fraction of predictions that match the true labels."""
    return sum(t == p for t, p in zip(true, predicted)) / len(true)

majority_class = Counter(gold_labels).most_common(1)[0][0]
baseline_acc = accuracy(gold_labels, [majority_class] * len(gold_labels)) * 100
v1_acc = accuracy(gold_labels, v1_predictions) * 100
v2_acc = accuracy(gold_labels, v2_predictions) * 100

print(f"Majority-class baseline (always '{majority_class}'): {baseline_acc:.1f}%")
print(f"Prompt V1 accuracy: {v1_acc:.1f}%  (margin: {v1_acc - baseline_acc:+.1f} pp)")
print(f"Prompt V2 accuracy: {v2_acc:.1f}%  (margin: {v2_acc - baseline_acc:+.1f} pp)")

Majority-class baseline (always 'billing'): 45.0%
Prompt V1 accuracy: 70.0%  (margin: +25.0 pp)
Prompt V2 accuracy: 80.0%  (margin: +35.0 pp)


## Step 3 — Confusion Matrices

Accuracy tells us *how many* errors each prompt makes. Confusion matrices tell us *what kind*. We'll build side-by-side matrices so we can see exactly where each prompt struggles.

In [4]:
def confusion_matrix(true, predicted, labels):
    """Build a confusion matrix: matrix[true][predicted] = count."""
    matrix = {t: {p: 0 for p in labels} for t in labels}
    for t, p in zip(true, predicted):
        matrix[t][p] += 1
    return matrix

def format_cm(cm, labels, title):
    """Return a markdown table for a confusion matrix."""
    header = f"#### {title}\n\n| True \\ Predicted | " + " | ".join(labels) + " |"
    sep = "|" + "---|" * (len(labels) + 1)
    rows = []
    for t in labels:
        vals = " | ".join(str(cm[t][p]) for p in labels)
        rows.append(f"| **{t}** | {vals} |")
    return header + "\n" + sep + "\n" + "\n".join(rows)

labels = sorted(set(gold_labels))  # ['account', 'billing', 'technical']
cm_v1 = confusion_matrix(gold_labels, v1_predictions, labels)
cm_v2 = confusion_matrix(gold_labels, v2_predictions, labels)

md(format_cm(cm_v1, labels, "Prompt V1 — Confusion Matrix") + "\n\n" +
   format_cm(cm_v2, labels, "Prompt V2 — Confusion Matrix") + "\n\n" +
   "*Rows = true label, Columns = predicted label. Diagonal = correct.*")

#### Prompt V1 — Confusion Matrix

| True \ Predicted | account | billing | technical |
|---|---|---|---|
| **account** | 5 | 3 | 1 |
| **billing** | 1 | 15 | 2 |
| **technical** | 2 | 3 | 8 |

#### Prompt V2 — Confusion Matrix

| True \ Predicted | account | billing | technical |
|---|---|---|---|
| **account** | 7 | 1 | 1 |
| **billing** | 2 | 14 | 2 |
| **technical** | 1 | 1 | 11 |

*Rows = true label, Columns = predicted label. Diagonal = correct.*

## Step 4 — Per-Class Precision and Recall

Recall from `mini-score-against-gold-set`:
- **Precision** = of everything predicted as class X, what fraction truly was X?
- **Recall** = of everything truly class X, what fraction did the system catch?

We compute these for both prompts and display them side by side.

In [5]:
def precision_recall(cm, labels):
    """Compute per-class precision and recall from a confusion matrix."""
    metrics = {}
    for cls in labels:
        tp = cm[cls][cls]
        predicted_as_cls = sum(cm[t][cls] for t in labels)
        actually_cls = sum(cm[cls][p] for p in labels)
        precision = tp / predicted_as_cls if predicted_as_cls > 0 else 0.0
        recall = tp / actually_cls if actually_cls > 0 else 0.0
        metrics[cls] = {"precision": precision, "recall": recall, "support": actually_cls}
    return metrics

m_v1 = precision_recall(cm_v1, labels)
m_v2 = precision_recall(cm_v2, labels)

# Side-by-side comparison table
rows = [
    "| Class | V1 Precision | V1 Recall | V2 Precision | V2 Recall | Support |",
    "|---|---|---|---|---|---|"
]
for cls in labels:
    rows.append(
        f"| **{cls}** "
        f"| {m_v1[cls]['precision']:.0%} | {m_v1[cls]['recall']:.0%} "
        f"| {m_v2[cls]['precision']:.0%} | {m_v2[cls]['recall']:.0%} "
        f"| {m_v1[cls]['support']} |"
    )

md("### Per-Class Precision & Recall — V1 vs V2\n\n" + "\n".join(rows))

### Per-Class Precision & Recall — V1 vs V2

| Class | V1 Precision | V1 Recall | V2 Precision | V2 Recall | Support |
|---|---|---|---|---|---|
| **account** | 62% | 56% | 70% | 78% | 9 |
| **billing** | 71% | 83% | 88% | 78% | 18 |
| **technical** | 73% | 62% | 79% | 85% | 13 |

## Step 5 — Latency and Cost Comparison

Quality alone doesn't determine whether to ship. Prompt V2 is longer (it includes examples), so it uses more tokens and may have higher latency. Let's measure that.

We use logged latency and token data, just as we did in `mini-latency-and-cost`.

In [6]:
import random
random.seed(99)

def simulate_perf_logs(n, avg_prompt_tok, latency_base):
    """Simulate performance logs for one prompt version."""
    logs = []
    for _ in range(n):
        if random.random() < 0.92:
            lat = random.uniform(latency_base * 0.7, latency_base * 1.3)
        else:
            lat = random.uniform(latency_base * 3, latency_base * 6)
        logs.append({
            "latency_ms": round(lat, 1),
            "prompt_tokens": random.randint(avg_prompt_tok - 40, avg_prompt_tok + 40),
            "completion_tokens": random.randint(5, 25),
        })
    return logs

# V1: shorter prompt → fewer tokens, lower latency
logs_v1 = simulate_perf_logs(40, avg_prompt_tok=180, latency_base=220)
# V2: longer prompt with examples → more tokens, higher latency
logs_v2 = simulate_perf_logs(40, avg_prompt_tok=310, latency_base=340)

def percentile(values, p):
    s = sorted(values)
    return s[min(int(len(s) * p / 100), len(s) - 1)]

def compute_stats(logs, label):
    lats = [e["latency_ms"] for e in logs]
    avg_pt = sum(e["prompt_tokens"] for e in logs) / len(logs)
    avg_ct = sum(e["completion_tokens"] for e in logs) / len(logs)
    return {
        "label": label,
        "p50": percentile(lats, 50),
        "p95": percentile(lats, 95),
        "avg_prompt_tokens": avg_pt,
        "avg_completion_tokens": avg_ct,
    }

v1_stats = compute_stats(logs_v1, "V1")
v2_stats = compute_stats(logs_v2, "V2")

# Cost calculation (gpt-4o-mini pricing)
PRICE_IN = 0.15   # $ per 1M input tokens
PRICE_OUT = 0.60  # $ per 1M output tokens

for stats in [v1_stats, v2_stats]:
    cost = (
        stats["avg_prompt_tokens"] * PRICE_IN / 1_000_000
        + stats["avg_completion_tokens"] * PRICE_OUT / 1_000_000
    )
    stats["cost_per_1000"] = cost * 1000

# Display comparison
md(f"""### Latency & Cost — V1 vs V2

| Metric | V1 | V2 |
|--------|-----|-----|
| p50 latency | {v1_stats['p50']:.0f} ms | {v2_stats['p50']:.0f} ms |
| p95 latency | {v1_stats['p95']:.0f} ms | {v2_stats['p95']:.0f} ms |
| Avg prompt tokens | {v1_stats['avg_prompt_tokens']:.0f} | {v2_stats['avg_prompt_tokens']:.0f} |
| Avg completion tokens | {v1_stats['avg_completion_tokens']:.0f} | {v2_stats['avg_completion_tokens']:.0f} |
| **Cost per 1,000** | **${v1_stats['cost_per_1000']:.4f}** | **${v2_stats['cost_per_1000']:.4f}** |
""")

### Latency & Cost — V1 vs V2

| Metric | V1 | V2 |
|--------|-----|-----|
| p50 latency | 246 ms | 332 ms |
| p95 latency | 974 ms | 1562 ms |
| Avg prompt tokens | 177 | 311 |
| Avg completion tokens | 15 | 15 |
| **Cost per 1,000** | **$0.0358** | **$0.0555** |


## Step 6 — Full Comparison Dashboard

Let's pull everything together — quality, latency, and cost — into one view. This is the evidence table that drives the ship decision.

In [7]:
dashboard = f"""### 📊 Prompt V1 vs V2 — Full Comparison

| Metric | Baseline | Prompt V1 | Prompt V2 |
|--------|----------|-----------|----------|
| **Accuracy** | {baseline_acc:.1f}% | {v1_acc:.1f}% | {v2_acc:.1f}% |
| Margin over baseline | — | {v1_acc - baseline_acc:+.1f} pp | {v2_acc - baseline_acc:+.1f} pp |
"""

# Add per-class rows
for cls in labels:
    dashboard += (
        f"| {cls} precision | — | {m_v1[cls]['precision']:.0%} | {m_v2[cls]['precision']:.0%} |\n"
        f"| {cls} recall | — | {m_v1[cls]['recall']:.0%} | {m_v2[cls]['recall']:.0%} |\n"
    )

dashboard += f"""
| **p50 latency** | — | {v1_stats['p50']:.0f} ms | {v2_stats['p50']:.0f} ms |
| **p95 latency** | — | {v1_stats['p95']:.0f} ms | {v2_stats['p95']:.0f} ms |
| **Cost per 1,000** | — | ${v1_stats['cost_per_1000']:.4f} | ${v2_stats['cost_per_1000']:.4f} |

*Gold set size: {len(gold_labels)} tickets. All metrics are directional estimates.*
"""

md(dashboard)

### 📊 Prompt V1 vs V2 — Full Comparison

| Metric | Baseline | Prompt V1 | Prompt V2 |
|--------|----------|-----------|----------|
| **Accuracy** | 45.0% | 70.0% | 80.0% |
| Margin over baseline | — | +25.0 pp | +35.0 pp |
| account precision | — | 62% | 70% |
| account recall | — | 56% | 78% |
| billing precision | — | 71% | 88% |
| billing recall | — | 83% | 78% |
| technical precision | — | 73% | 79% |
| technical recall | — | 62% | 85% |

| **p50 latency** | — | 246 ms | 332 ms |
| **p95 latency** | — | 974 ms | 1562 ms |
| **Cost per 1,000** | — | $0.0358 | $0.0555 |

*Gold set size: 40 tickets. All metrics are directional estimates.*


## Step 7 — Failure Mode Check

Before writing the ship decision, recall from `mini-failure-mode-prediction` that we should ask: **how could each prompt break silently in production?**

Let's check whether either prompt produced any hallucinated labels (labels outside the allowed set) or refusals during our evaluation run.

In [8]:
VALID_LABELS = {"billing", "technical", "account"}

def check_silent_failures(predictions, version_name):
    """Check for hallucinated or invalid labels."""
    invalid = [p for p in predictions if p not in VALID_LABELS]
    if invalid:
        print(f"⚠️  {version_name}: {len(invalid)} invalid label(s) found: {invalid}")
    else:
        print(f"✅  {version_name}: all {len(predictions)} predictions are valid labels")
    return invalid

check_silent_failures(v1_predictions, "Prompt V1")
check_silent_failures(v2_predictions, "Prompt V2")

print("\n💡 No hallucinated labels in this run, but in production you should")
print("   validate every response against the allowed label set (see mini-failure-mode-prediction).")

✅  Prompt V1: all 40 predictions are valid labels
✅  Prompt V2: all 40 predictions are valid labels

💡 No hallucinated labels in this run, but in production you should
   validate every response against the allowed label set (see mini-failure-mode-prediction).


## Step 8 — Write the Ship Decision

An **evidence-based ship decision** is not "V2 looks better, let's ship it." It's a structured document that:

1. **States the question** clearly
2. **Presents the evidence** (the metrics we just computed)
3. **Acknowledges the trade-offs** — what gets better AND what gets worse
4. **Makes a recommendation** with its accepted risks
5. **States the caveats** (sample size, conditions under which to revisit)

This is **prompt versioning** in action: we treat the prompt change like a code change and gate it behind evidence.

In [9]:
# Calculate key comparison values for the decision
v2_tech_recall_gain = m_v2["technical"]["recall"] - m_v1["technical"]["recall"]
v2_acct_recall_gain = m_v2["account"]["recall"] - m_v1["account"]["recall"]
v2_billing_recall_loss = m_v2["billing"]["recall"] - m_v1["billing"]["recall"]
cost_increase_pct = (
    (v2_stats["cost_per_1000"] - v1_stats["cost_per_1000"]) / v1_stats["cost_per_1000"] * 100
)

decision = f"""### 📋 Ship Decision: Prompt V1 → V2

**Question:** Should we replace Prompt V1 with Prompt V2 for support-ticket classification?

---

#### Evidence Summary

| Metric | V1 | V2 | Change |
|--------|----|----|--------|
| Overall accuracy | {v1_acc:.1f}% | {v2_acc:.1f}% | {v2_acc - v1_acc:+.1f} pp |
| technical recall | {m_v1['technical']['recall']:.0%} | {m_v2['technical']['recall']:.0%} | {v2_tech_recall_gain:+.0%} |
| account recall | {m_v1['account']['recall']:.0%} | {m_v2['account']['recall']:.0%} | {v2_acct_recall_gain:+.0%} |
| billing recall | {m_v1['billing']['recall']:.0%} | {m_v2['billing']['recall']:.0%} | {v2_billing_recall_loss:+.0%} |
| p50 latency | {v1_stats['p50']:.0f} ms | {v2_stats['p50']:.0f} ms | +{v2_stats['p50'] - v1_stats['p50']:.0f} ms |
| Cost per 1,000 | ${v1_stats['cost_per_1000']:.4f} | ${v2_stats['cost_per_1000']:.4f} | +{cost_increase_pct:.0f}% |

Both versions beat the majority-class baseline ({baseline_acc:.1f}%) by a clear margin.

#### What Gets Better with V2

- **Technical recall jumps from {m_v1['technical']['recall']:.0%} to {m_v2['technical']['recall']:.0%}** — V1 was misclassifying {13 - cm_v1['technical']['technical']} of 13 technical tickets; V2 only misses {13 - cm_v2['technical']['technical']}.
- **Account recall improves from {m_v1['account']['recall']:.0%} to {m_v2['account']['recall']:.0%}** — fewer account tickets lost to billing.
- Overall accuracy rises by {v2_acc - v1_acc:.1f} percentage points.

#### What Gets Worse with V2

- **Billing recall drops from {m_v1['billing']['recall']:.0%} to {m_v2['billing']['recall']:.0%}** — one extra billing ticket misclassified.
- **Latency increases** — p50 goes from {v1_stats['p50']:.0f} ms to {v2_stats['p50']:.0f} ms (the longer prompt takes more time).
- **Cost rises ~{cost_increase_pct:.0f}%** — more prompt tokens per call.

#### Accepted Trade-offs

- The small billing-recall drop ({v2_billing_recall_loss:+.0%}) is acceptable because billing is already the largest class and still has the highest recall.
- The cost increase is modest in absolute terms (${v2_stats['cost_per_1000'] - v1_stats['cost_per_1000']:.4f} per 1,000 records) and worth the quality gain.
- Latency stays well under 1 second at p50, which is acceptable for a background classification system.

#### Recommendation

**Ship V2.** It meaningfully improves recall on the two minority classes (technical, account) with only a minor trade-off on billing recall and a modest cost increase. The system adds clear value over the baseline for all three classes.

#### Caveats

- ⚠️ Gold set is only {len(gold_labels)} rows — all metrics are directional estimates.
- Expand to 80–100 rows before treating these differences as statistically significant.
- Monitor billing recall in production for the first week after switching to V2.
"""

md(decision)

### 📋 Ship Decision: Prompt V1 → V2

**Question:** Should we replace Prompt V1 with Prompt V2 for support-ticket classification?

---

#### Evidence Summary

| Metric | V1 | V2 | Change |
|--------|----|----|--------|
| Overall accuracy | 70.0% | 80.0% | +10.0 pp |
| technical recall | 62% | 85% | +23% |
| account recall | 56% | 78% | +22% |
| billing recall | 83% | 78% | -6% |
| p50 latency | 246 ms | 332 ms | +86 ms |
| Cost per 1,000 | $0.0358 | $0.0555 | +55% |

Both versions beat the majority-class baseline (45.0%) by a clear margin.

#### What Gets Better with V2

- **Technical recall jumps from 62% to 85%** — V1 was misclassifying 5 of 13 technical tickets; V2 only misses 2.
- **Account recall improves from 56% to 78%** — fewer account tickets lost to billing.
- Overall accuracy rises by 10.0 percentage points.

#### What Gets Worse with V2

- **Billing recall drops from 83% to 78%** — one extra billing ticket misclassified.
- **Latency increases** — p50 goes from 246 ms to 332 ms (the longer prompt takes more time).
- **Cost rises ~55%** — more prompt tokens per call.

#### Accepted Trade-offs

- The small billing-recall drop (-6%) is acceptable because billing is already the largest class and still has the highest recall.
- The cost increase is modest in absolute terms ($0.0197 per 1,000 records) and worth the quality gain.
- Latency stays well under 1 second at p50, which is acceptable for a background classification system.

#### Recommendation

**Ship V2.** It meaningfully improves recall on the two minority classes (technical, account) with only a minor trade-off on billing recall and a modest cost increase. The system adds clear value over the baseline for all three classes.

#### Caveats

- ⚠️ Gold set is only 40 rows — all metrics are directional estimates.
- Expand to 80–100 rows before treating these differences as statistically significant.
- Monitor billing recall in production for the first week after switching to V2.


---
## Self-check rubric

Tag `v4` only after the report passes this list.


## What Makes a Ship Decision Defensible?

In `lab-prompt-ab-evaluation`, you built a full evaluation pipeline and wrote a ship decision at the end. This mini-lab zooms in on **the decision itself** — what must it contain, and how do you check that nothing is missing?

A ship decision is not "it looks good, let's deploy." It's a **self-check** — a structured document where every claim is backed by a number, every trade-off is named, and every limitation is stated honestly.

Think of it as a checklist you run *on your own evaluation* before presenting it to anyone.

## Step 1 — The Evaluation Checklist

Before writing a ship decision, verify that you have all the required evidence. Here's the checklist — every item must have a concrete value, not just "we checked this."

In [1]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

checklist = """
### ✅ Evaluation Self-Check Checklist

| # | Evidence Required | Why It Matters | Your Value |
|---|---|---|---|
| 1 | **Gold set size** | Determines how much you can trust your metrics | ___ rows |
| 2 | **Baseline accuracy** | Proves the AI adds value over a dumb predictor | ___% |
| 3 | **System accuracy** | Overall correctness on the gold set | ___% |
| 4 | **Per-class precision & recall** | Reveals where the system fails (accuracy hides this) | per class |
| 5 | **p50 latency** | Typical user experience | ___ ms |
| 6 | **p95 latency** | Worst reasonable user experience | ___ ms |
| 7 | **Cost per 1,000 records** | Is this viable at production scale? | $___ |
| 8 | **Prompt / model version** | Which exact configuration produced these results? | v___ |
| 9 | **Trade-offs accepted** | What gets worse and why that's OK | stated |
| 10 | **Sample-size caveat** | Honest statement of confidence limits | stated |

> **Rule:** If any row is blank, the ship decision is not ready.
"""
md(checklist)


### ✅ Evaluation Self-Check Checklist

| # | Evidence Required | Why It Matters | Your Value |
|---|---|---|---|
| 1 | **Gold set size** | Determines how much you can trust your metrics | ___ rows |
| 2 | **Baseline accuracy** | Proves the AI adds value over a dumb predictor | ___% |
| 3 | **System accuracy** | Overall correctness on the gold set | ___% |
| 4 | **Per-class precision & recall** | Reveals where the system fails (accuracy hides this) | per class |
| 5 | **p50 latency** | Typical user experience | ___ ms |
| 6 | **p95 latency** | Worst reasonable user experience | ___ ms |
| 7 | **Cost per 1,000 records** | Is this viable at production scale? | $___ |
| 8 | **Prompt / model version** | Which exact configuration produced these results? | v___ |
| 9 | **Trade-offs accepted** | What gets worse and why that's OK | stated |
| 10 | **Sample-size caveat** | Honest statement of confidence limits | stated |

> **Rule:** If any row is blank, the ship decision is not ready.


## Step 2 — A Filled-In Example

Let's fill in the checklist with concrete numbers from an evaluation run. We'll use results similar to what you saw in `lab-prompt-ab-evaluation` — a support-ticket classifier being evaluated for a prompt upgrade.

In [2]:
# Evaluation results from a completed run
eval_results = {
    "gold_set_size": 40,
    "baseline_accuracy": 45.0,      # majority-class baseline
    "system_accuracy": 80.0,         # prompt V2
    "per_class": {
        "billing":   {"precision": 0.82, "recall": 0.78},
        "technical": {"precision": 0.79, "recall": 0.85},
        "account":   {"precision": 0.78, "recall": 0.78},
    },
    "p50_latency_ms": 330,
    "p95_latency_ms": 1850,
    "cost_per_1000": 0.0554,
    "prompt_version": "V2",
    "model": "gpt-4o-mini",
}

# Fill in the checklist
r = eval_results
pc = r["per_class"]

precision_recall_rows = ""
for cls in ["billing", "technical", "account"]:
    precision_recall_rows += (
        f"| | {cls} | P={pc[cls]['precision']:.0%}  R={pc[cls]['recall']:.0%} |\n"
    )

filled = f"""
### ✅ Evaluation Self-Check — Filled In

| # | Evidence Required | Value |
|---|---|---|
| 1 | **Gold set size** | {r['gold_set_size']} rows |
| 2 | **Baseline accuracy** | {r['baseline_accuracy']:.1f}% (majority-class) |
| 3 | **System accuracy** | {r['system_accuracy']:.1f}% |
| 4 | **Margin over baseline** | +{r['system_accuracy'] - r['baseline_accuracy']:.1f} pp |
{precision_recall_rows}\
| 5 | **p50 latency** | {r['p50_latency_ms']} ms |
| 6 | **p95 latency** | {r['p95_latency_ms']} ms |
| 7 | **Cost per 1,000** | ${r['cost_per_1000']:.4f} |
| 8 | **Version** | Prompt {r['prompt_version']}, model `{r['model']}` |

All rows filled → we can proceed to the ship decision.
"""
md(filled)


### ✅ Evaluation Self-Check — Filled In

| # | Evidence Required | Value |
|---|---|---|
| 1 | **Gold set size** | 40 rows |
| 2 | **Baseline accuracy** | 45.0% (majority-class) |
| 3 | **System accuracy** | 80.0% |
| 4 | **Margin over baseline** | +35.0 pp |
| | billing | P=82%  R=78% |
| | technical | P=79%  R=85% |
| | account | P=78%  R=78% |
| 5 | **p50 latency** | 330 ms |
| 6 | **p95 latency** | 1850 ms |
| 7 | **Cost per 1,000** | $0.0554 |
| 8 | **Version** | Prompt V2, model `gpt-4o-mini` |

All rows filled → we can proceed to the ship decision.


## Step 3 — Write the Ship Decision

A defensible ship decision has **five parts**:

1. **Question** — What are you deciding?
2. **Evidence** — The metrics (not opinions)
3. **What gets better / worse** — Trade-offs named explicitly
4. **Recommendation** — Ship, don't ship, or conditional
5. **Caveats** — Sample size, conditions to revisit

Let's generate one from the results above.

In [3]:
r = eval_results
pc = r["per_class"]

decision = f"""### 📋 Ship Decision: Deploy Prompt {r['prompt_version']} with `{r['model']}`

**Question:** Is the support-ticket classifier ready to deploy with Prompt {r['prompt_version']}?

---

#### Evidence

| Metric | Value |
|--------|-------|
| Gold set | {r['gold_set_size']} rows |
| Baseline accuracy | {r['baseline_accuracy']:.1f}% |
| System accuracy | {r['system_accuracy']:.1f}% (+{r['system_accuracy'] - r['baseline_accuracy']:.1f} pp over baseline) |
| billing P/R | {pc['billing']['precision']:.0%} / {pc['billing']['recall']:.0%} |
| technical P/R | {pc['technical']['precision']:.0%} / {pc['technical']['recall']:.0%} |
| account P/R | {pc['account']['precision']:.0%} / {pc['account']['recall']:.0%} |
| p50 latency | {r['p50_latency_ms']} ms |
| p95 latency | {r['p95_latency_ms']} ms |
| Cost per 1,000 | ${r['cost_per_1000']:.4f} |
| Configuration | Prompt {r['prompt_version']}, `{r['model']}` |

#### What Gets Better

- System accuracy is **{r['system_accuracy'] - r['baseline_accuracy']:.1f} pp above baseline** — the classifier adds clear value over random guessing.
- All three classes have recall ≥ 78%, meaning no category is being systematically missed.
- Cost is low at ${r['cost_per_1000']:.4f} per 1,000 records (${r['cost_per_1000'] * 100:.2f} per 100K).

#### What Gets Worse / Accepted Trade-offs

- **p95 latency is {r['p95_latency_ms']} ms** — about 1 in 20 requests takes nearly 2 seconds. This is acceptable for a background classification job but would need investigation for a user-facing feature.
- **Billing precision at {pc['billing']['precision']:.0%}** means ~18% of tickets predicted as billing are actually something else. Acceptable for routing (human reviews follow), but not for automated billing actions.

#### Recommendation

**Ship** for background ticket routing. The system beats the baseline by a wide margin across all classes. Latency and cost are within acceptable bounds for a non-real-time use case.

#### Caveats

- ⚠️ Gold set is only **{r['gold_set_size']} rows** — metrics are directional estimates. Expand to 80–100 rows before treating precision/recall differences as reliable.
- Monitor billing precision in the first week of production. If it drops below 75%, revisit the prompt.
- These results are tied to **Prompt {r['prompt_version']}** on **`{r['model']}`**. Changing either requires re-evaluation.
"""

md(decision)

### 📋 Ship Decision: Deploy Prompt V2 with `gpt-4o-mini`

**Question:** Is the support-ticket classifier ready to deploy with Prompt V2?

---

#### Evidence

| Metric | Value |
|--------|-------|
| Gold set | 40 rows |
| Baseline accuracy | 45.0% |
| System accuracy | 80.0% (+35.0 pp over baseline) |
| billing P/R | 82% / 78% |
| technical P/R | 79% / 85% |
| account P/R | 78% / 78% |
| p50 latency | 330 ms |
| p95 latency | 1850 ms |
| Cost per 1,000 | $0.0554 |
| Configuration | Prompt V2, `gpt-4o-mini` |

#### What Gets Better

- System accuracy is **35.0 pp above baseline** — the classifier adds clear value over random guessing.
- All three classes have recall ≥ 78%, meaning no category is being systematically missed.
- Cost is low at $0.0554 per 1,000 records ($5.54 per 100K).

#### What Gets Worse / Accepted Trade-offs

- **p95 latency is 1850 ms** — about 1 in 20 requests takes nearly 2 seconds. This is acceptable for a background classification job but would need investigation for a user-facing feature.
- **Billing precision at 82%** means ~18% of tickets predicted as billing are actually something else. Acceptable for routing (human reviews follow), but not for automated billing actions.

#### Recommendation

**Ship** for background ticket routing. The system beats the baseline by a wide margin across all classes. Latency and cost are within acceptable bounds for a non-real-time use case.

#### Caveats

- ⚠️ Gold set is only **40 rows** — metrics are directional estimates. Expand to 80–100 rows before treating precision/recall differences as reliable.
- Monitor billing precision in the first week of production. If it drops below 75%, revisit the prompt.
- These results are tied to **Prompt V2** on **`gpt-4o-mini`**. Changing either requires re-evaluation.


## Step 4 — Spot the Missing Evidence

Now let's practice the critical skill: **catching when a ship decision is incomplete.** Below is a decision report with deliberate gaps. Can the checklist catch them?

In [4]:
# An incomplete evaluation report — on purpose
incomplete_report = {
    "gold_set_size": None,           # never stated!
    "baseline_accuracy": None,       # never computed!
    "system_accuracy": 82.0,
    "per_class": None,               # only overall accuracy reported
    "p50_latency_ms": 280,
    "p95_latency_ms": None,          # only p50 reported
    "cost_per_1000": 0.0480,
    "prompt_version": None,          # "the latest prompt" — which one?
    "model": "gpt-4o-mini",
}

# Run the self-check
checks = [
    ("Gold set size",           incomplete_report["gold_set_size"]),
    ("Baseline accuracy",       incomplete_report["baseline_accuracy"]),
    ("System accuracy",         incomplete_report["system_accuracy"]),
    ("Per-class precision/recall", incomplete_report["per_class"]),
    ("p50 latency",             incomplete_report["p50_latency_ms"]),
    ("p95 latency",             incomplete_report["p95_latency_ms"]),
    ("Cost per 1,000",          incomplete_report["cost_per_1000"]),
    ("Prompt version",          incomplete_report["prompt_version"]),
]

print("Self-Check Results:\n")
missing = 0
for name, value in checks:
    if value is None:
        print(f"  ❌ {name}: MISSING")
        missing += 1
    else:
        print(f"  ✅ {name}: {value}")

print(f"\n{'='*40}")
if missing > 0:
    print(f"🚫 {missing} item(s) missing — this evaluation is NOT ready for a ship decision.")
else:
    print("✅ All items present — ready to write the ship decision.")

Self-Check Results:

  ❌ Gold set size: MISSING
  ❌ Baseline accuracy: MISSING
  ✅ System accuracy: 82.0
  ❌ Per-class precision/recall: MISSING
  ✅ p50 latency: 280
  ❌ p95 latency: MISSING
  ✅ Cost per 1,000: 0.048
  ❌ Prompt version: MISSING

🚫 5 item(s) missing — this evaluation is NOT ready for a ship decision.


**Why each gap matters:**

- **No gold set size** → We can't assess confidence. 82% on 10 rows means very little; 82% on 200 rows is solid.
- **No baseline** → 82% accuracy sounds good, but if the majority class is 80%, the AI barely beats random guessing.
- **No per-class metrics** → Overall 82% could mean 95% on billing and 40% on account. You'd never know.
- **No p95 latency** → p50 of 280 ms looks fast, but if p95 is 5,000 ms, 1 in 20 users waits 5 seconds.
- **No prompt version** → If something breaks, you can't reproduce the evaluation or roll back. This is **version control** — knowing exactly which prompt + model produced which results.

## Step 5 — Version Control: Tag Your Evaluation

Every ship decision should record **exactly** which configuration produced the results. This isn't about Git commits (though those help) — it's about the minimum version metadata that makes an evaluation reproducible.

In [5]:
from datetime import datetime

eval_tag = {
    "eval_id": "eval-2024-001",
    "date": datetime.now().strftime("%Y-%m-%d"),
    "prompt_version": "V2",
    "model": "gpt-4o-mini",
    "gold_set_version": "gold-v1 (40 rows)",
    "evaluator": "your-name",
    "decision": "SHIP for background routing",
}

tag_rows = "\n".join(f"| **{k}** | {v} |" for k, v in eval_tag.items())

md(f"""### 🏷️ Evaluation Tag

Save this alongside your ship decision so anyone can trace the results back to the exact setup.

| Field | Value |
|-------|-------|
{tag_rows}

> **Tip:** Store this tag in a file (e.g., `eval-2024-001.json`) or in your project's version control.
> If the prompt or model changes, a new evaluation with a new tag is required.
""")

### 🏷️ Evaluation Tag

Save this alongside your ship decision so anyone can trace the results back to the exact setup.

| Field | Value |
|-------|-------|
| **eval_id** | eval-2024-001 |
| **date** | 2026-09-16 |
| **prompt_version** | V2 |
| **model** | gpt-4o-mini |
| **gold_set_version** | gold-v1 (40 rows) |
| **evaluator** | your-name |
| **decision** | SHIP for background routing |

> **Tip:** Store this tag in a file (e.g., `eval-2024-001.json`) or in your project's version control.
> If the prompt or model changes, a new evaluation with a new tag is required.


## 🎯 Summary

### What You Built

You built a complete prompt A/B evaluation pipeline that compares two prompt versions against the same gold set, measures quality (accuracy, per-class precision and recall, confusion matrices), latency (p50/p95), and cost — then synthesizes the evidence into a defensible ship/don't-ship recommendation.

### Key Takeaways

1. **Prompt versioning** — treat prompt changes like code changes: never swap prompts in production without running both versions against the same gold set and comparing the evidence
2. **Baseline comparison** — both prompt versions must beat the majority-class baseline; if they don't, the AI system isn't adding value regardless of which version "wins"
3. **Per-class metrics over accuracy** — overall accuracy can hide that one prompt is much better at minority classes; precision and recall per class reveal *where* each version succeeds and fails
4. **Quality + latency + cost** — a better prompt that's too slow or too expensive might not be worth shipping; always evaluate all three dimensions together
5. **Evidence-based decision** — a ship recommendation states the question, presents the evidence, acknowledges what gets worse, names the accepted trade-offs, and flags the sample-size caveats